# RL-Chess-Engine: does a non-zero draw target break the draw cycle?

The three-seed rerun (`seeds_rerun.ipynb`, technical report §5.8) found that training inside the
draw cycle leaves the network *worse* than its random start. The proposed mechanism: almost no
self-play game ends in checkmate, so almost every value target is 0, the value head stays flat,
and the policy learns to sharpen its own preferences, shuffles included.

A local diagnostic ruled out exploration as the lever: with a trained rerun network, sampling
moves for 60 plies instead of 15 left 11 of 12 games in threefold repetition, and sampling for
the whole game only swapped repetitions for games that hit the 200-ply cap, which also score 0.

This notebook changes one thing from the rerun's baseline: a self-play game that ends without
checkmate gets its value target from the final material balance
(`TrainingConfig.draw_value_target = "material"`, about +0.8 for a queen up) instead of 0.
Same init seeds (1, 2, 3) and settings, so each network is compared with the same starting
networks (`docs/data/seeds_rerun_2026-11.json`) and with the rerun's baseline.

About 3 × 1.5 h on one T4. The repo is cloned outside `/kaggle/working`, so the output is a
single small zip.

In [ ]:
ARM = "material"
FLAGS = {"material": "--draw-target material",
         "explore-material": "--temperature-plies 60 --draw-target material"}[ARM]
SEEDS = [1, 2, 3]

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 https://github.com/Danny-397/RL-Chess-Engine.git /tmp/RL-Chess-Engine
%cd /tmp/RL-Chess-Engine
!pip install -q python-chess onnx onnxscript
!git log --oneline -1

## Train and evaluate each seed (baseline settings, one change)

In [ ]:
for s in SEEDS:
    !python run_experiment.py --device cuda --skip-scaled --skip-assisted --seed {s} --tag {ARM} {FLAGS}

## Value probe on every trained network

In [ ]:
cks = ' '.join(f'--checkpoint experiment_out/seed{s}-{ARM}/baseline.pt' for s in SEEDS)
!python tools/value_probe.py --no-untrained {cks} --out experiment_out/value_probe_{ARM}.json

## Package (no checkpoint directories; the three final networks are kept)

In [ ]:
!cd experiment_out && zip -qr /kaggle/working/chess_{ARM}.zip . -x '*/checkpoints/*'
!ls -la /kaggle/working/